# Task 3 — PostgreSQL Database
## Store & Query Processed Reviews

This notebook:
1. Installs and verifies PostgreSQL is running
2. Creates the `bank_reviews` database
3. Creates the `banks` and `reviews` tables
4. Inserts all analyzed reviews from Task 2
5. Runs verification queries to confirm data integrity


## 1. PostgreSQL Setup

Run these commands **in your terminal** (not in this notebook) before continuing.

### Install PostgreSQL (Ubuntu / WSL)
```bash
sudo apt update
sudo apt install postgresql postgresql-contrib -y
```

### Start the PostgreSQL service
```bash
sudo service postgresql start
```

### Verify it is running
```bash
sudo service postgresql status
```
You should see: `Active: active (running)`

### Create the database and set a password
```bash
sudo -u postgres psql
```
Then inside the psql shell:
```sql
ALTER USER postgres PASSWORD 'yourpassword';
CREATE DATABASE bank_reviews;
\q
```

Once done, come back here and continue with Cell 2.


## 2. Imports & Configuration

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import pandas as pd
from src.database import (
    get_connection,
    create_tables,
    insert_banks,
    insert_reviews,
    verify_insertion,
    theme_breakdown,
)

# ── Database credentials ───────────────────────────────────────────────────────
# Change password to match what you set in the terminal above
DB_CONFIG = {
    "dbname":   "bank_reviews",
    "user":     "postgres",
    "password": "yourpassword",   # <-- change this
    "host":     "localhost",
    "port":     5432,
}

# ── Input file from Task 2 ─────────────────────────────────────────────────────
ANALYZED_PATH = "../data/raw/reviews_analyzed.csv"

print("Configuration ready.")

## 3. Load Analyzed Reviews

Load the CSV produced by Task 2. This file must have all 8 columns:
`review, rating, date, bank, source, sentiment_label, sentiment_score, identified_theme`

In [ ]:
df = pd.read_csv(ANALYZED_PATH)

print(f"Shape   : {df.shape}")
print(f"Columns : {list(df.columns)}")
print(f"\nReviews per bank:")
print(df["bank"].value_counts().to_string())
print(f"\nMissing values:")
print(df.isnull().sum().to_string())
df.head()

## 4. Connect to PostgreSQL

If this cell throws an error:
- Make sure PostgreSQL is running: `sudo service postgresql start`
- Make sure the password in DB_CONFIG matches what you set
- Make sure the `bank_reviews` database was created

In [ ]:
conn = get_connection(**DB_CONFIG)
print("Connection successful.")

## 5. Create Tables

Creates two tables:

**`banks`** — one row per bank (parent table)
| Column | Type | Description |
|--------|------|-------------|
| bank_id | SERIAL PRIMARY KEY | Auto-generated unique ID |
| bank_name | VARCHAR UNIQUE | CBE, BOA, Dashen |
| app_id | VARCHAR | Google Play app package name |
| country | VARCHAR | et (Ethiopia) |

**`reviews`** — one row per review (child table)
| Column | Type | Description |
|--------|------|-------------|
| review_id | SERIAL PRIMARY KEY | Auto-generated unique ID |
| bank_id | INT → banks.bank_id | Foreign key linking to parent bank |
| review_text | TEXT | Raw review content |
| rating | INT (1–5) | Star rating with CHECK constraint |
| review_date | DATE | Normalized to YYYY-MM-DD |
| sentiment_label | VARCHAR | positive / negative |
| sentiment_score | FLOAT | DistilBERT confidence (0–1) |
| identified_theme | VARCHAR | Business theme from Task 2 |
| source | VARCHAR | Google Play |
| created_at | TIMESTAMP | Row insertion timestamp |

### Why a foreign key?
Storing `bank_id` (an integer) in every review row is far more efficient
than storing the full bank name string. It also enforces referential integrity —
you cannot insert a review for a bank that does not exist in the banks table.

In [ ]:
create_tables(conn)

## 6. Insert Banks

Insert the three banks into the `banks` table.
`ON CONFLICT DO NOTHING` makes this safe to re-run.

In [ ]:
bank_ids = insert_banks(conn)
print(f"Bank ID mapping: {bank_ids}")

## 7. Insert Reviews

Bulk-insert all reviews using `execute_values` — this sends all rows
in a single SQL statement instead of one per row, making it ~50x faster.

In [ ]:
inserted = insert_reviews(conn, df, bank_ids)
print(f"\nTotal rows inserted: {inserted}")

## 8. Verify Insertion

Run SQL queries to confirm the data loaded correctly.
Compare these numbers against your Task 1 and Task 2 outputs.

In [ ]:
summary = verify_insertion(conn)
print("=== Summary per bank ===")
print(summary.to_string(index=False))

In [ ]:
themes = theme_breakdown(conn)
print("=== Theme breakdown per bank ===")
print(themes.to_string(index=False))

## 9. Explore with Raw SQL

Run SQL queries directly to answer business questions.
These are the kinds of queries you will reference in your Task 4 report.

In [ ]:
import pandas as pd

# Query 1: Average rating per bank
q1 = pd.read_sql("""
    SELECT b.bank_name,
           ROUND(AVG(r.rating)::NUMERIC, 2) AS avg_rating,
           COUNT(*) AS total_reviews
    FROM reviews r
    JOIN banks b ON r.bank_id = b.bank_id
    GROUP BY b.bank_name
    ORDER BY avg_rating DESC;
""", conn)

print("Q1 — Average rating per bank:")
print(q1.to_string(index=False))

In [ ]:
# Query 2: Most common themes for negative reviews only
q2 = pd.read_sql("""
    SELECT b.bank_name,
           r.identified_theme,
           COUNT(*) AS negative_count
    FROM reviews r
    JOIN banks b ON r.bank_id = b.bank_id
    WHERE r.sentiment_label = 'negative'
    GROUP BY b.bank_name, r.identified_theme
    ORDER BY b.bank_name, negative_count DESC;
""", conn)

print("Q2 — Top negative themes per bank:")
print(q2.to_string(index=False))

In [ ]:
# Query 3: Monthly review trend per bank
q3 = pd.read_sql("""
    SELECT b.bank_name,
           TO_CHAR(r.review_date, 'YYYY-MM') AS month,
           COUNT(*) AS review_count,
           ROUND(AVG(r.rating)::NUMERIC, 2) AS avg_rating
    FROM reviews r
    JOIN banks b ON r.bank_id = b.bank_id
    GROUP BY b.bank_name, month
    ORDER BY b.bank_name, month;
""", conn)

print("Q3 — Monthly trend:")
print(q3.to_string(index=False))

In [ ]:
# Query 4: 1-star reviews only — what are users most angry about?
q4 = pd.read_sql("""
    SELECT b.bank_name,
           r.identified_theme,
           COUNT(*) AS one_star_count
    FROM reviews r
    JOIN banks b ON r.bank_id = b.bank_id
    WHERE r.rating = 1
    GROUP BY b.bank_name, r.identified_theme
    ORDER BY b.bank_name, one_star_count DESC;
""", conn)

print("Q4 — 1-star review themes:")
print(q4.to_string(index=False))

## 10. Close Connection

Always close the connection when you are done.

In [ ]:
conn.close()
print("Connection closed.")

## 11. Schema Reference

```
banks
─────────────────────────────────
bank_id    SERIAL PRIMARY KEY
bank_name  VARCHAR(100) UNIQUE
app_id     VARCHAR(200)
country    VARCHAR(10)

        │  (one bank → many reviews)
        │
        ▼

reviews
─────────────────────────────────
review_id        SERIAL PRIMARY KEY
bank_id          INT → banks.bank_id   (FK)
review_text      TEXT
rating           INT  CHECK (1–5)
review_date      DATE
sentiment_label  VARCHAR(20)
sentiment_score  FLOAT
identified_theme VARCHAR(100)
source           VARCHAR(50)
created_at       TIMESTAMP
```
